In [5]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
from CMAPSSDataset import *
import CMAPSSDataset as cmap
import time

print(f"XGBoost version: {xgb.__version__}")

XGBoost version: 3.2.0


### Construcción de features por ventana

In [3]:
def construir_features_xgb(df, sensor_cols, window_size=30):
    """
    Para cada fila (ciclo) construye un vector de features que resume
    la ventana de los últimos window_size ciclos.

    Features por sensor:
      - Valor actual
      - Delta acumulado desde ciclo 1
      - Slope local
      - Aceleración
      - Media, std, min, max, rango sobre la ventana
      - Slope de regresión lineal sobre la ventana completa
      - Último valor menos primer valor de la ventana (tendencia corta)

    Features globales:
      - Ciclo actual (normalizado por motor)
      - Condición operacional
    """
    registros = []

    for motor_id, grupo in df.groupby("motor_id"):
        grupo = grupo.reset_index(drop=True)
        n_ciclos  = len(grupo)
        max_ciclo = grupo["ciclo"].max()

        for t in range(n_ciclos):
            inicio  = max(0, t - window_size + 1)
            ventana = grupo.iloc[inicio:t + 1]

            fila = {}

            # ── Features globales del ciclo ──────────────────────────────
            #fila["ciclo_actual"]   = grupo.loc[t, "ciclo"]
            #fila["ciclo_relativo"] = grupo.loc[t, "ciclo"] / max_ciclo DATA LEAKAGE eliminado
            fila["condicion"]      = grupo.loc[t, "condicion"] \
                                     if "condicion" in grupo.columns else 0

            # ── Configuración operacional actual ─────────────────────────
            for op in ["op_1", "op_2", "op_3"]:
                if op in grupo.columns:
                    fila[op] = grupo.loc[t, op]

            # ── Features por sensor ──────────────────────────────────────
            for col in sensor_cols:
                v = ventana[col].values

                # Valor actual y derivadas en t
                fila[f"{col}_actual"]  = grupo.loc[t, col]
                fila[f"{col}_delta"]   = grupo.loc[t, f"{col}_delta"] \
                                         if f"{col}_delta" in grupo.columns \
                                         else v[-1] - v[0]
                fila[f"{col}_slope"]   = grupo.loc[t, f"{col}_slope"] \
                                         if f"{col}_slope" in grupo.columns else 0
                fila[f"{col}_accel"]   = grupo.loc[t, f"{col}_accel"] \
                                         if f"{col}_accel" in grupo.columns else 0

                # Estadísticas sobre la ventana
                fila[f"{col}_mean"]    = v.mean()
                fila[f"{col}_std"]     = v.std() if len(v) > 1 else 0
                fila[f"{col}_min"]     = v.min()
                fila[f"{col}_max"]     = v.max()
                fila[f"{col}_rango"]   = v.max() - v.min()
                #fila[f"{col}_trend"]   = stats.linregress(np.arange(len(v)), v).slope

                # Diferencia entre último y primer ciclo de la ventana
                fila[f"{col}_delta_ventana"] = v[-1] - v[0]

            fila["rul"]      = grupo.loc[t, "rul"]
            fila["motor_id"] = motor_id
            registros.append(fila)

    return pd.DataFrame(registros)

### Preparar Dataset

In [4]:
def preparar_dataset_xgb(ruta_base, subset, window_size=60):
    config        = cmap.CONFIG_DATASETS[subset]
    n_condiciones = config["n_condiciones"]
    rul_max       = config["rul_max"]

    COLS = (
        ["motor_id", "ciclo"] +
        [f"op_{i}" for i in range(1, 4)] +
        [f"s{i}"   for i in range(1, 22)]
    )
    train = pd.read_csv(f"{ruta_base}/train_{subset}.txt", sep=r"\s+", header=None, names=COLS)
    test  = pd.read_csv(f"{ruta_base}/test_{subset}.txt",  sep=r"\s+", header=None, names=COLS)
    rul   = pd.read_csv(f"{ruta_base}/RUL_{subset}.txt",   sep=r"\s+", header=None, names=["rul_final"])

    train_df = cmap.add_rul_train(train.copy(), rul_max=rul_max)
    test_df  = cmap.add_rul_test(test.copy(),   rul, rul_max=rul_max)

    # 1. Eliminar constantes (sobre train)
    sensor_cols = [c for c in train_df.columns if c.startswith("s")]
    std         = train_df[sensor_cols].std()
    constantes  = std[std < 0.01].index.tolist()
    train_df    = train_df.drop(columns=constantes)
    test_df     = test_df.drop(columns=constantes)
    sensor_cols = [c for c in sensor_cols if c not in constantes]

    # 2. SPLIT POR MOTOR — antes de cualquier fit
    motores = train_df["motor_id"].unique()
    gss     = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    idx_tr, idx_val = next(gss.split(motores, groups=motores))
    motores_train = motores[idx_tr]
    motores_val   = motores[idx_val]

    train_solo = train_df[train_df["motor_id"].isin(motores_train)].copy()
    val_solo   = train_df[train_df["motor_id"].isin(motores_val)].copy()

    print(f"Motores únicos en train: {len(motores_train)}")
    print(f"Motores únicos en val:   {len(motores_val)}")

    # 3. KMeans — fit solo sobre train_solo
    if n_condiciones > 1:
        train_solo, km = cmap.identificar_condicion_operacional(train_solo, n_condiciones)
        val_solo["condicion"]  = km.predict(val_solo[["op_1","op_2","op_3"]])
        test_df["condicion"]   = km.predict(test_df[["op_1","op_2","op_3"]])
    else:
        train_solo["condicion"] = 0
        val_solo["condicion"]   = 0
        test_df["condicion"]    = 0

    # 4. Features de degradación
    print(f"[{subset}] Calculando features de degradación...")
    deg_cols   = [c for c in train_solo.columns if c.startswith("s")]
    train_solo = cmap.añadir_features_degradacion_rapido(train_solo, deg_cols)
    val_solo   = cmap.añadir_features_degradacion_rapido(val_solo,   deg_cols)
    test_df    = cmap.añadir_features_degradacion_rapido(test_df,    deg_cols)

    # 5. Normalización — fit en train_solo, transform en val y test
    feature_cols = [c for c in train_solo.columns if c.startswith("s") or c.startswith("op")]
    train_solo, val_solo, test_df, scalers = cmap.normalizar_por_condicion(
        train_solo, val_solo, test_df, feature_cols, n_condiciones
    )

    # 6. Tabularización — features agregadas por ventana
    print(f"[{subset}] Construyendo features agregadas...")
    train_feat = construir_features_xgb(train_solo, sensor_cols, window_size)
    val_feat   = construir_features_xgb(val_solo,   sensor_cols, window_size)
    test_feat  = construir_features_xgb(test_df,    sensor_cols, window_size)

    # Test: solo el último ciclo de cada motor
    test_feat = test_feat.groupby("motor_id").last().reset_index()

    xgb_feature_cols = [c for c in train_feat.columns
                        if c not in ["rul", "motor_id", "ciclo_actual"]]

    X_train = train_feat[xgb_feature_cols].values.astype(np.float32)
    y_train = train_feat["rul"].values.astype(np.float32)
    X_val   = val_feat[xgb_feature_cols].values.astype(np.float32)
    y_val   = val_feat["rul"].values.astype(np.float32)
    X_test  = test_feat[xgb_feature_cols].values.astype(np.float32)
    y_test  = test_feat["rul"].values.astype(np.float32)

    print(f"[{subset}] Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")
    print(f"[{subset}] Features XGB: {len(xgb_feature_cols)} | Condiciones: {n_condiciones}")

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), scalers, xgb_feature_cols

### Entrenamiento XGBoost

In [6]:
def nasa_score_numpy(y_true, y_pred):
    diff = y_pred - y_true
    return np.sum(np.where(diff < 0,
                           np.exp(-diff / 13.0) - 1,
                           np.exp( diff / 10.0) - 1))

def entrenar_xgb(subset, ruta_base="CMAPSSData", window_size=60,
                 learning_rate=0.05, early_stopping=50):

    cfg = cmap.CONFIG_DATASETS[subset]
    print(f"Parámetros de {subset}: rul_max = {cfg['rul_max']} | "
          f"condiciones = {cfg['n_condiciones']}")

    # Datos
    t0 = time.time()
    (X_tr, y_tr), (X_val, y_val), (X_te, y_te), scalers, feature_cols = \
        preparar_dataset_xgb(ruta_base, subset, window_size)
    t_datos = time.time() - t0

    w_tr = np.where(y_tr < 30, 4.0, np.where(y_tr < 80, 2.0, 1.0))

    model = xgb.XGBRegressor(
        n_estimators          = 3000,
        learning_rate         = learning_rate,
        max_depth             = 6,
        min_child_weight      = 5,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        objective             = "reg:squarederror",
        eval_metric           = "rmse",
        early_stopping_rounds = early_stopping,
        device                = "cuda",
        random_state          = 42,
    )

    t0 = time.time()
    model.fit(
        X_tr, y_tr,
        sample_weight = w_tr,
        eval_set      = [(X_val, y_val)],
        verbose       = 100
    )
    t_train = time.time() - t0

    print(f"[{subset}] Tiempo preprocesando: {t_datos/60:.2f} min")
    print(f"[{subset}] Tiempo entrenando:    {t_train/60:.2f} min")
    print(f"[{subset}] Árboles usados: {model.best_iteration + 1}/3000")

    return model, feature_cols, (X_te, y_te)


def evaluar_xgb(model, X_test, y_test, subset):
    y_pred = model.predict(X_test)
    diff   = y_pred - y_test

    print(f"% predicciones tardías:  {(diff > 0).mean()*100:.1f}%")
    print(f"Error medio tardío:      {diff[diff > 0].mean():.1f}")
    print(f"Error medio temprano:    {diff[diff < 0].mean():.1f}")

    score = np.where(diff < 0,
                     np.exp(-diff / 13.0) - 1,
                     np.exp( diff / 10.0) - 1)

    print(f"\n── {subset} ──────────────────────────")
    print(f"RMSE  : {np.sqrt(np.mean(diff**2)):.2f}")
    print(f"MAE   : {np.mean(np.abs(diff)):.2f}")
    print(f"Score : {score.sum():.1f}")

    return y_pred

### Evaluar XGB

In [8]:
RUTA = "../CMAPSSData"
resultados_xgb = {}

for subset in ["FD001", "FD002", "FD003", "FD004"]:
    print(f"\n{'='*50}\n{subset}")
    model, feats, (X_te, y_te) = entrenar_xgb(subset, ruta_base=RUTA)
    y_pred = evaluar_xgb(model, X_te, y_te, subset)
    resultados_xgb[subset] = {
        "model": model, "feats": feats,
        "y_test": y_te, "y_pred": y_pred
    }



FD001
Parámetros de FD001: rul_max = 125 | condiciones = 1
Motores únicos en train: 80
Motores únicos en val:   20
[FD001] Calculando features de degradación...
[FD001] Construyendo features agregadas...
[FD001] Train (16561, 144) | Val (4070, 144) | Test (100, 144)
[FD001] Features XGB: 144 | Condiciones: 1
[0]	validation_0-rmse:45.52130
[100]	validation_0-rmse:13.56326
[200]	validation_0-rmse:13.44682
[300]	validation_0-rmse:13.41953
[400]	validation_0-rmse:13.40336
[500]	validation_0-rmse:13.38822
[509]	validation_0-rmse:13.38947
[FD001] Tiempo preprocesando: 1.53 min
[FD001] Tiempo entrenando:    0.11 min
[FD001] Árboles usados: 460/3000
% predicciones tardías:  57.0%
Error medio tardío:      11.2
Error medio temprano:    -10.7

── FD001 ──────────────────────────
RMSE  : 15.50
MAE   : 11.01
Score : 535.2

FD002
Parámetros de FD002: rul_max = 125 | condiciones = 6


/home/marco/miniconda3/envs/tfm_torch/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [19:17:40] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1772125072520/work/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Motores únicos en train: 208
Motores únicos en val:   52
[FD002] Calculando features de degradación...
[FD002] Construyendo features agregadas...
[FD002] Train (43464, 204) | Val (10295, 204) | Test (259, 204)
[FD002] Features XGB: 204 | Condiciones: 6
[0]	validation_0-rmse:45.04283
[100]	validation_0-rmse:16.34829
[158]	validation_0-rmse:16.37278
[FD002] Tiempo preprocesando: 5.79 min
[FD002] Tiempo entrenando:    0.04 min
[FD002] Árboles usados: 109/3000
% predicciones tardías:  41.3%
Error medio tardío:      9.2
Error medio temprano:    -12.0

── FD002 ──────────────────────────
RMSE  : 14.56
MAE   : 10.86
Score : 762.3

FD003
Parámetros de FD003: rul_max = 150 | condiciones = 1
Motores únicos en train: 80
Motores únicos en val:   20
[FD003] Calculando features de degradación...
[FD003] Construyendo features agregadas...
[FD003] Train (20012, 154) | Val (4708, 154) | Test (100, 154)
[FD003] Features XGB: 154 | Condiciones: 1
[0]	validation_0-rmse:53.54833
[100]	validation_0-rmse:17.

### Interpretación de features

In [ ]:
def plot_feature_importance(model, feature_cols, subset, top_n=20):
    """
    XGBoost permite ver exactamente qué features importan más.
    Esto es oro para la sección de discusión del paper:
    puedes argumentar qué sensores son más predictivos de degradación.
    """
    importancias = model.feature_importances_
    idx          = np.argsort(importancias)[::-1][:top_n]

    plt.figure(figsize=(10, 6))
    plt.barh(range(top_n),
             importancias[idx][::-1],
             color="steelblue", alpha=0.8)
    plt.yticks(range(top_n),
               [feature_cols[i] for i in idx][::-1],
               fontsize=9)
    plt.xlabel("Importancia")
    plt.title(f"{subset} — Top {top_n} features más predictivas")
    plt.tight_layout()
    plt.show()

    print(f"\nTop 10 features — {subset}:")
    for i in idx[:10]:
        print(f"  {feature_cols[i]:35s}  {importancias[i]:.4f}")

for subset, res in resultados_xgb.items():
    plot_feature_importance(
        res["model"], res["feats"], subset, top_n=20)